# Customer Radar — huấn luyện ViSoBERT v5 (L3 gốc rễ gia dụng + loại hàng + số sao)

**Chuẩn bị trên máy:**
1. Chạy `cd backend && npm run nlp:prepare` để cập nhật `nlp_service/labels.json` có đầu `root` (18 mã L3 `gia_dung`).
2. Xuất bộ huấn luyện chống rò rỉ theo sản phẩm & Jaccard >= 0,8:
   `cd backend && node scripts/export_training_data.js --dbs=customer_radar_real,cr_t_6b063cbc5122d551a58f9737`
   → `nlp_service/data/real/{train,dev,test}.jsonl` + `manifest.json`.
3. Tạo Kaggle Dataset `customer-radar-v5` chứa thư mục `nlp_service/` (mã nguồn + `labels.json` + `data/`; KHÔNG đưa `checkpoints/` hay `.venv/`).
4. Notebook: GPU T4, Internet On, Add Data = `customer-radar-v5`.

**Đầu ra `/kaggle/working/`:**
- `ckpt-v5.zip`: checkpoint PyTorch v5 đã hiệu chỉnh ngưỡng `rootThresholds` trên dev.
- `onnx-v5.zip`: bản ONNX INT8 (chạy trình duyệt & Hugging Face Space CPU).
- `test_report_v5.json`: **con số được phép báo cáo** trên gold test (kèm L2/L3 macro-F1, từng lớp L3, ma trận nhầm, tỉ lệ hành động được, KTC 95% bootstrap theo sản phẩm).

In [ ]:
import glob, os, shutil, subprocess, json
src = os.path.dirname(glob.glob("/kaggle/input/**/nlp_service/train.py", recursive=True)[0])
shutil.copytree(src, "/kaggle/working/nlp_service", dirs_exist_ok=True)
os.chdir("/kaggle/working/nlp_service")
print(open("data/real/manifest.json", encoding="utf-8").read())

In [ ]:
# Trọng số gốc ViSoBERT (đã xin phép tác giả — trích dẫn Nguyen và cs., EMNLP 2023)
from huggingface_hub import snapshot_download
base = snapshot_download("uitnlp/visobert", local_dir="/kaggle/working/visobert")
print("Base:", base)

In [ ]:
# Huấn luyện v5: nhãn người (1,0) + thầy Qwen3 trùng 2 lượt (0,5) + Claude (0,5); đầu vào [N sao] [loại hàng] văn bản
train_files = ["data/real/train.jsonl"] + [p for p in ["data/visfd_train.jsonl"] if os.path.exists(p)]
dev = "data/real/dev.jsonl"
cmd = [
    "python", "train.py",
    "--train", *train_files,
    "--dev", dev,
    "--exclude", "data/real/test.jsonl",
    "--base", "/kaggle/working/visobert",
    "--out", "/kaggle/working/ckpt-v5",
    "--epochs", "5",
    "--batch-size", "32",
    "--max-len", "160",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Hiệu chỉnh nhiệt độ và ngưỡng từng lớp L1/L2/L3 trên tập dev (KHÔNG đụng test)
subprocess.run([
    "python", "calibrate.py",
    "--checkpoint", "/kaggle/working/ckpt-v5",
    "--dev", "data/real/dev.jsonl",
], check=True)

In [ ]:
# Đánh giá cuối cùng trên GOLD TEST giữ riêng (KTC 95% bootstrap theo sản phẩm)
subprocess.run([
    "python", "eval_real.py",
    "--checkpoint", "/kaggle/working/ckpt-v5",
    "--test", "data/real/test.jsonl",
    "--out", "/kaggle/working/test_report_v5.json",
], check=True)

In [ ]:
# Xuất ONNX FP32 + INT8 (~98 MB) và đóng gói tải về
subprocess.run(["pip", "install", "-q", "onnx", "onnxruntime"], check=True)
subprocess.run([
    "python", "deploy/export_onnx.py",
    "--checkpoint", "/kaggle/working/ckpt-v5",
    "--out", "/kaggle/working/onnx-v5",
], check=True)
shutil.make_archive("/kaggle/working/ckpt-v5", "zip", "/kaggle/working/ckpt-v5")
shutil.make_archive("/kaggle/working/onnx-v5", "zip", "/kaggle/working/onnx-v5")
print("Tệp đầu ra:", sorted(os.listdir("/kaggle/working")))